# <font color="steelblue">30. Comparar dos grupos: proporciones, tablas y medias</font>

*Unidad 5. Conceptos básicos de inferencia estadística* · Sesión teórica 30 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`, `salarios_redaccion.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv", "salarios_redaccion.csv"]
PREGUNTAS = ["q_P30_1.json", "q_P30_2.json", "q_P30_3.json", "q_P30_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

¿Creen más bulos quienes comparten noticias? ¿Cobran distinto mujeres y hombres en la redacción? ¿Es real la diferencia entre dos provincias? Casi todas las preguntas periodísticas con datos acaban siendo **comparaciones entre grupos**. En este último cuaderno juntarás todo lo aprendido para contestarlas con rigor y redactar el resultado.

**Qué necesitas saber antes**: los cuadernos 12, 27 y 29.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Comparar dos proporciones con un intervalo y un contraste.
2. Contrastar la independencia en una tabla de contingencia (chi-cuadrado).
3. Comparar dos medias con la t de Student.
4. Redactar una comparación con su incertidumbre.

## <font color='steelblue'>Contenidos</font>

1. Dos proporciones
2. Tablas de contingencia: la prueba chi-cuadrado
3. Dos medias: la t de Student
4. Cómo contarlo: lista de comprobación final

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Dos proporciones</font>

In [ ]:
enc = pd.read_csv("encuesta_medios.csv")
sal = pd.read_csv("salarios_redaccion.csv")

Pregunta: ¿es mayor la proporción de personas que han creído un bulo entre quienes comparten noticias?

El intervalo para la **diferencia de proporciones** es $(\hat{p}_1 - \hat{p}_2) \pm 1{,}96\sqrt{\frac{\hat{p}_1(1-\hat{p}_1)}{n_1} + \frac{\hat{p}_2(1-\hat{p}_2)}{n_2}}$.

In [ ]:
g1 = enc[enc["comparte_noticias"] == "Sí"]["ha_creido_bulo"] == "Sí"
g2 = enc[enc["comparte_noticias"] == "No"]["ha_creido_bulo"] == "Sí"
p1, p2, n1, n2 = g1.mean(), g2.mean(), len(g1), len(g2)
dif = p1 - p2
ee = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
print(f"Comparten: {p1:.1%} (n = {n1})   No comparten: {p2:.1%} (n = {n2})")
print(f"Diferencia: {dif * 100:.1f} puntos, IC 95 %: de {(dif - 1.96 * ee) * 100:.1f} a {(dif + 1.96 * ee) * 100:.1f}")

El intervalo está muy lejos de 0: la diferencia es real (en esta muestra simulada). El contraste correspondiente ($H_0$: $p_1 = p_2$) da un p-valor minúsculo:

In [ ]:
p_comun = (g1.sum() + g2.sum()) / (n1 + n2)
z = dif / np.sqrt(p_comun * (1 - p_comun) * (1 / n1 + 1 / n2))
print(f"z = {z:.2f}, p-valor (bilateral) = {2 * (1 - stats.norm.cdf(abs(z))):.2e}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P30_1.json")

## <font color='steelblue'>2. Tablas de contingencia: la prueba chi-cuadrado</font>

Con más de dos categorías se usa la **prueba chi-cuadrado de independencia**: compara la tabla observada con la que esperaríamos **si las variables fueran independientes** (cuaderno 21).

- $H_0$: las dos variables son independientes.

In [ ]:
tabla = pd.crosstab(enc["provincia"], enc["ha_creido_bulo"])
chi2, p, gl, esperadas = stats.chi2_contingency(tabla)
print(tabla)
print(f"\nChi-cuadrado = {chi2:.2f}, grados de libertad = {gl}, p-valor = {p:.3f}")
print("Frecuencias esperadas si fueran independientes:")
print(pd.DataFrame(esperadas, index=tabla.index, columns=tabla.columns).round(1))

**Cómo leerlo**: en el cuaderno 21 Castellón parecía tener más personas que habían creído un bulo (33 % frente a 25–26 %). Pero el p-valor (0,16) es mayor que 0,05: con solo 160 personas de Castellón en la muestra, esa diferencia es compatible con el azar. Titular «Castellón, la provincia más crédula» sería un error.

Ahora, una relación clara:

In [ ]:
tabla2 = pd.crosstab(enc["medio_principal"], enc["comparte_noticias"])
print(f"Medio principal y compartir noticias: p-valor = {stats.chi2_contingency(tabla2)[1]:.2e}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P30_2.json")

## <font color='steelblue'>3. Dos medias: la t de Student</font>

Para comparar las medias de una variable cuantitativa en dos grupos se usa la **prueba t** (en la versión de Welch, que no supone varianzas iguales). Comparemos el salario de mujeres y hombres **dentro del mismo puesto** (redactores y redactoras), para no mezclar puestos distintos.

In [ ]:
red = sal[sal["puesto"] == "Redactor/a"]
m = red.loc[red["sexo"] == "Mujer", "salario_bruto_anual"]
h = red.loc[red["sexo"] == "Hombre", "salario_bruto_anual"]
res = stats.ttest_ind(h, m, equal_var=False)
dif = h.mean() - m.mean()
ee = np.sqrt(h.var() / len(h) + m.var() / len(m))
print(f"Hombres: {h.mean():,.0f} € (n = {len(h)})   Mujeres: {m.mean():,.0f} € (n = {len(m)})")
print(f"Diferencia: {dif:,.0f} €, IC 95 % aprox.: de {dif - 1.96 * ee:,.0f} a {dif + 1.96 * ee:,.0f} €")
print(f"p-valor = {res.pvalue:.4f}")

**Cómo leerlo**: los redactores cobran de media unos 1.700 € más que las redactoras. El p-valor está justo por debajo de 0,05 y el intervalo va de casi 0 a unos 3.400 €: hay indicios de diferencia, pero la estimación es muy imprecisa. Con datos reales habría que tener en cuenta también la experiencia y el tipo de contrato.

In [ ]:
red.boxplot(column="salario_bruto_anual", by="sexo", vert=False, grid=False)
plt.suptitle("")
plt.title("Salario de redactores y redactoras (datos simulados)")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P30_3.json")

## <font color='steelblue'>4. Cómo contarlo: lista de comprobación final</font>

Antes de publicar una comparación con datos, repasa esta lista. Resume todo el curso:

1. **¿De dónde salen los datos?** Fuente, fecha y ficha técnica (cuadernos 1 y 25).
2. **¿Qué se compara exactamente?** Grupos comparables, mismo periodo, misma definición (8 y 30).
3. **¿Media o mediana? ¿Puntos o porcentaje?** (8 y 14).
4. **¿Sobre quién se calcula el porcentaje?** (5, 12 y 21).
5. **¿Cuántos casos hay en cada grupo y cuál es el margen de error?** (27 y 28).
6. **¿Puede ser azar?** Intervalo de confianza o p-valor (29 y 30).
7. **¿Es grande la diferencia, además de significativa?** (29).
8. **¿Correlación o causa?** ¿Hay variables de confusión? (13).
9. **¿Es estacional o real?** ¿Está en euros reales? (16 y 19).
10. **¿El gráfico es honesto?** Eje, escala, título y fuente (6).

In [ ]:
print(f"Entre las personas que comparten noticias, el {p1:.0%} reconoce haber creído alguna vez un bulo,")
print(f"frente al {p2:.0%} de quienes no comparten (encuesta simulada a {len(enc)} personas; diferencia de")
print(f"{(p1 - p2) * 100:.0f} puntos, con un margen de error de ±{1.96 * np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2) * 100:.0f} puntos).")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y marca **todas** las respuestas correctas:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P30_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

¿Pagan por noticias en mayor proporción las personas con estudios universitarios que el resto? Calcula la diferencia y su intervalo.

<details><summary>Ver solución</summary>

```python
u = enc["estudios"] == "Universitarios"
a = enc.loc[u, "suscripcion_pago"] == "Sí"
b = enc.loc[~u, "suscripcion_pago"] == "Sí"
d = a.mean() - b.mean()
e = 1.96 * np.sqrt(a.mean() * (1 - a.mean()) / len(a) + b.mean() * (1 - b.mean()) / len(b))
print(d, d - e, d + e)
```
</details>

### **Ejercicio 2** *(sección 2)*

Haz la prueba chi-cuadrado entre el nivel de estudios y haber creído un bulo.

<details><summary>Ver solución</summary>

```python
stats.chi2_contingency(pd.crosstab(enc["estudios"], enc["ha_creido_bulo"]))[1]
```
</details>

### **Ejercicio 3** *(sección 3)*

Compara el salario medio de personas con contrato indefinido y temporal. ¿Es significativa la diferencia?

<details><summary>Ver solución</summary>

```python
i = sal.loc[sal["contrato"] == "Indefinido", "salario_bruto_anual"]
t = sal.loc[sal["contrato"] == "Temporal", "salario_bruto_anual"]
print(i.mean() - t.mean(), stats.ttest_ind(i, t, equal_var=False).pvalue)
```
</details>

### **Ejercicio 4** *(sección 4)*

**Trabajo final (como el de la evaluación continua)**: elige una comparación con los datos del curso, compruébala con la lista de la sección 4 y escribe una pieza de 250 palabras con un gráfico honesto y la frase metodológica.

<details><summary>Ver idea clave</summary>

La guía docente evalúa: claridad periodística, contenido estadístico riguroso y creatividad en la elección del estadístico y los parámetros.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- **Dos proporciones**: intervalo para la diferencia; si no incluye el 0, hay evidencia de diferencia.
- **Chi-cuadrado** (`stats.chi2_contingency`): contrasta la independencia en una tabla de contingencia.
- **Dos medias**: prueba t de Welch (`stats.ttest_ind(..., equal_var=False)`), comparando grupos **comparables**.
- Publica la diferencia **con su incertidumbre** (margen o intervalo) y el número de casos.
- Usa la **lista de comprobación** antes de publicar cualquier comparación con datos.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Diferencia de proporciones** | Resta de las proporciones de dos grupos. |
| **Chi-cuadrado** | Prueba de independencia entre variables cualitativas. |
| **Frecuencias esperadas** | Recuentos que habría en una tabla si las variables fueran independientes. |
| **Prueba t de Welch** | Prueba para comparar dos medias sin suponer varianzas iguales. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

**¡Enhorabuena!** Has terminado el curso. Para seguir: practica con datos reales del [INE](https://www.ine.es), del [CIS](https://www.cis.es) y de [datos.gob.es](https://datos.gob.es), y revisa los cuadernos de SIMUSTAT para ampliar (regresión múltiple, ANOVA).